## Splitting corpora to train, dev and test sets

In [2]:
import os
from estnltk.converters import json_to_text
import sklearn
from estnltk.converters import text_to_json

Kui suure osa kõigist ajafaktide korpuse lausetest moodustavad riigikogu stenogrammid?

In [3]:
tempf_corpus_path = 'C:/Users/liivas/Documents/Magistritoo/andmetootlus/corpus_preprocessing/temporal_facts_corpus_json/'

news_articles_size = 0
rkogu_articles_size = 0

for filename in os.listdir(tempf_corpus_path):
    text_obj = json_to_text(file=os.path.join(tempf_corpus_path, filename))
    if 'rkogu' in text_obj.meta['filename']:
        rkogu_articles_size+=len(text_obj.sentences)
    else:
        news_articles_size+=len(text_obj.sentences)

In [14]:
print(f'Ajaleheartiklite suurus lausetes: {news_articles_size}')
print(f'Riigikogu stenogrammide suurus lausetes: {rkogu_articles_size}')
print(f'Riigikogu stenogrammide osakaal kõigist korpuse lausetest: {rkogu_articles_size/(news_articles_size+rkogu_articles_size)}')

Ajaleheartiklite suurus lausetes: 2540
Riigikogu stenogrammide suurus lausetes: 528
Riigikogu stenogrammide osakaal kõigist korpuse lausetest: 0.17209908735332463


Before model training, data from both corpora needs to be divided into train, development and test sets. Train set will include ~80%, development set ~10% and test set ~10% of corpus data. Data will be splitted on document level, but the approach to data division will be different for each corpus.

In case on TimeML corpus, article sizes in words will be considered. The articles for each set are chosen randomly, but the split generation step will be repeated with different seeds until test set size in words is approximately 10% of the size of the whole corpus.

In case on temporal facts corpus, article sizes in sentences will be considered, the reason being that temporal facts corpus has significally less annotatations, compared to TimeML corpus. The articles for each set are chosen randomly, but the split generation step will be repeated with different seeds until test set size in sentences is approximately 10% of the size of the whole corpus.

NB! parliament transcripts (rkogu) and historical articles (horisont) will be omitted. These can be used separately later, for additional model performance evaluations.

In [15]:
timeml_corpus_path = 'C:/Users/liivas/Documents/Magistritoo/andmetootlus/corpus_preprocessing/EstTimeML_corpus_json/'

timeml_article_texts = []

for filename in os.listdir(timeml_corpus_path):
    text_obj = json_to_text(file=os.path.join(timeml_corpus_path, filename))
    timeml_article_texts.append(text_obj)

timeml_article_texts[0]

Text(text='20.11.2002 Tallinna Sadam ehitab Saaremaa süvasadama Küdema lahe läänerannikule endisesse Tamme sadamakohta Mustjala valda . 70 mln kr maksev sadam tahetakse valmis saada 2004. aasta maiks . Hooaja jooksul peaks sadam vastu võtma 30-35 kruiisilaeva . Tallinna Sadama juhtkond põhjendas sadama ehitamist Tammele väikseimate süvenduskuludega , samuti pole vaja sadama maa-ala kelleltki välja osta , sest see kuulub riigile . Tammel oli kalasadam veel 1960. aastate lõpul , enne uue kalasadama rajamist Tagalahte Veerele . Saare maavanem Jüri Saar ütles , et looduslikult on Tamme ilusaim sadamakoht nii maalt kui merelt vaadatuna . Maavanem pakub selle Põhja-Saaremaale ehitatava sadama nimeks selguse mõttes Saaremaa sadam . Mustjala vallavanem Enno Kolter ütles , et uskus juba kaheksa aastat tagasi sadama tulevat just Tammele . Uudepanga lahte tahtis süvasadamat ehitada juba 2000. aasta suveks AS Undva Sadam . Selle vastu olid aga eelkõige linnuteadlased , sest Uudepanga lahes asub ohustatud linnu kirjuhaha suurim talvitusala Läänemeres . Toonane AS Undva Sadama nõukogu esimees Jaak Lokk ei imesta , et süvasadam nüüd lõpuks ikkagi ehitatakse . " Džinn oli ju pudelist välja lastud . Varem või hiljem tulnuks see sadam teha , sest muidu hakatakse küsima , miks te ühe projekti põhja lasite , aga asemele midagi ei pakkunud , " ütles Lokk . Ornitoloogid on pidanud parimaks sadamakohaks Veeret , mis praegu kuulub AS Hiiu Kalurile .')

In [4]:
tempf_corpus_path = 'C:/Users/liivas/Documents/Magistritoo/andmetootlus/corpus_preprocessing/temporal_facts_corpus_json/'

tempf_news = []
tempf_rkogu = []
tempf_horisont = []

for filename in os.listdir(tempf_corpus_path):
    text_obj = json_to_text(file=os.path.join(tempf_corpus_path, filename))
    if 'rkogu'  in text_obj.meta['filename']:
        tempf_rkogu.append(text_obj)
    elif 'horisont' in text_obj.meta['filename']:
        tempf_horisont.append(text_obj)
    else:
        tempf_news.append(text_obj)

tempf_news[0]

Text(text='Üleeilne päev oli ilmselt üks rõõmsamaid lehelugejatele ja masendavamaid ajakirjandusele .\nKümned tuhanded tallinlased leidsid sel päeval postkastist läikpaberil teate , et nad saavad ajalehte Postimees tellida üle nelja korra odavamalt kui näiteks tartlased .\nNii reetis Postimees oma kõige ustavamad toetajad : Tartu ja tartlased .\nEnt tasuta lõunaid kahjuks pole ja varem või hiljem peab lugeja ikka kauba eest maksma .\nPostimehe pöörase allahindluse taga on soov suretada teised lehed välja ja saavutada monopoolne seisund .\nSiis saab kergeusklikult tellijalt võtta mitmekordselt tagasi summa , mis neile praegu kingitakse .\nSelle asemel , et kulutada raha parema lehe tegemiseks ning pakkuda tellijale õige hinna eest võimalikult head kaupa , kulutab Postimees raha konkurentide väljasuretamiseks altvöölöökidega .\nSee ei ole ainuüksi rumal , vaid ka ohtlik , sest ajakirjandusmonopol ohustab meie kõigi sõnavabadust .\nLoomulikult ei saa Eesti Päevalehe toimetus , käed rüpes , pealt vaadata , kuidas Postimehe omanik , kes pole kordagi Eestis käinud , püüab Tallinnas tekitada samasugust ühe lehe umbset tõemonopoli , nagu see on neil korda läinud Tartus .\nMeil ei olnud valikut , me olime sunnitud kaitsma oma lugeja vabadust valida lehte sisu , mitte hinna järgi .\nSeepärast pakume kuni 22. juunini Tallinna , Harjumaa ja Tartu elanikele Eesti Päevalehe nelja kuu tellimust 60 krooni eest ( seni 288 krooni ) ja kuue kuu tellimust 95 krooni eest ( seni 432 krooni ) .\nMe saame aru , et see on ebaõiglane teiste piirkondade inimeste ja seniste , lehe eest täishinda maksnud tellijate suhtes , kuid niisuguse sundkäigu surus meile peale Postimehe alatus .\nEbaõigluse leevendamiseks otsustasime neil , kellel on 31. juuli seisuga tellitud Eesti Päevaleht täishinnaga , pikendada tellimust tasuta ühe kuu võrra kogu Eestis .\nTellija ei pea ise midagi tegema , toimetus saadab talle pärast tellimuse lõppu automaatselt veel kuu aja jooksul iga päev lehe koju .\nPostimehe vallandatud hinnasõda , mis lugejale võib esmapilgul tunduda suure õnnena , on tegelikult meie kõigi ühine õnnetus .\nEesti Päevalehe toimetus maksab üliodavate tellimuste ja juba olemasolevate tellimuste tasuta pikendamise eest soolast hinda ja ei saa seetõttu võibolla ellu viia kõiki plaane , mis olid kavandatud lehe arendamiseks .\nEnt mingis mõttes on see hind siiski madal , sest sõnavabadus ja lugeja valikuvabadus on hindamatu väärtus .\n')

#### Data division methods

In [ ]:
def get_size_by_strategy(texts, strategy):
    texts_size = 0
    
    if strategy == 'word_count':
        for text in texts:
            texts_size+=len(text.words)
    elif strategy == 'sentence_count':
        for text in texts:
            texts_size+=len(text.sentences)
    else:
        raise ValueError('Unknown strategy')
        
    return texts_size


def split_data(article_texts, strategy, random_state):
    train_size = int(len(article_texts)*0.8)
    dev_size = (len(article_texts)-train_size)//2
    test_size = len(article_texts)-(train_size+dev_size)
    
    article_texts_shuffled = sklearn.utils.shuffle(article_texts, random_state=random_state)
    
    train_texts = article_texts_shuffled[:train_size]
    dev_texts = article_texts_shuffled[-(dev_size+test_size):(-test_size)]
    test_texts = article_texts_shuffled[(-test_size):]
    
    assert len(train_texts) == train_size, 'Difference between train_size and actual train set size'
    assert len(dev_texts) == dev_size, 'Difference between dev_size and actual development set size'
    assert len(test_texts) == test_size, 'Difference between test_size and actual test set size'
    
    corpus_size = get_size_by_strategy(article_texts, strategy)
    test_size_by_strat = get_size_by_strategy(test_texts, strategy)
    
    test_size_percent = (test_size_by_strat/corpus_size)*100
    
    return train_texts, dev_texts, test_texts, test_size_percent

#### Splitting TimeML corpus data

In [18]:
timeml_train = None
timeml_dev = None
timeml_test = None
timeml_test_size_percent = None

for i in range(10):
    timeml_train, timeml_dev, timeml_test, timeml_test_size_percent = split_data(timeml_article_texts, 'word_count', i)
    if timeml_test_size_percent > 9 and timeml_test_size_percent < 11:
        print(f'Using seed {i}, the resulting TimeMLCorpus test size percent is {timeml_test_size_percent}')
        break
        
print()
# checking TimeMLCorpus train, dev and test sizes
print(f'Final TimeMLCorpus train size in articles: {len(timeml_train)}')
print(f'Final TimeMLCorpus dev size in articles: {len(timeml_dev)}')
print(f'Final TimeMLCorpus test size in articles: {len(timeml_test)}')

Using seed 1, the resulting TimeMLCorpus test size percent is 9.537794243889214

Final TimeMLCorpus train size in articles: 64
Final TimeMLCorpus dev size in articles: 8
Final TimeMLCorpus test size in articles: 8


#### Splitting temporal facts corpus data

In [ ]:
tempf_train = None
tempf_dev = None
tempf_test = None
tempf_test_size_percent = None

for i in range(10):
    tempf_train, tempf_dev, tempf_test, tempf_test_size_percent = split_data(tempf_news, 'sentence_count', i)
    if tempf_test_size_percent > 9 and tempf_test_size_percent < 11:
        print(f'Using seed {i}, the resulting temp. facts corpus news articles test size percent is {tempf_test_size_percent}')
        break
        
print(f"Final temp. facts corpus test size percent is {get_size_by_strategy(tempf_test, 'sentence_count')/get_size_by_strategy(tempf_train+tempf_dev+tempf_test, 'sentence_count')*100}")
print()
# checking final temporal facts corpus train, dev and test sizes
print(f'Final temp. facts corpus train size in articles: {len(tempf_train)}')
print(f'Final temp. facts corpus dev size in articles: {len(tempf_dev)}')
print(f'Final temp. facts corpus test size in articles: {len(tempf_test)}')

Using seed 0, the resulting temp. facts corpus news articles test size percent is 10.08849557522124
Final temp. facts corpus test size percent is 10.08849557522124

Final temp. facts corpus train size in articles: 61
Final temp. facts corpus dev size in articles: 8
Final temp. facts corpus test size in articles: 8


### Saving divided data

In [20]:
timeml_target_dir = 'model_data/TimeML'

# -- new directories for TimeML train, dev and test articles
timeml_train_path = os.path.join(timeml_target_dir, 'train')
timeml_dev_path = os.path.join(timeml_target_dir, 'dev')
timeml_test_path = os.path.join(timeml_target_dir, 'test')

#os.mkdir(timeml_train_path)
#os.mkdir(timeml_dev_path)
#os.mkdir(timeml_test_path)

for text in timeml_train:
    filename = os.path.join(timeml_train_path, text.meta['filename']) + '.json'
    text_to_json(text, file=filename)
    
for text in timeml_dev:
    filename = os.path.join(timeml_dev_path, text.meta['filename']) + '.json'
    text_to_json(text, file=filename)
    
for text in timeml_test:
    filename = os.path.join(timeml_test_path, text.meta['filename']) + '.json'
    text_to_json(text, file=filename)

In [10]:
tempf_target_dir = 'model_data/TempFact'

# -- new directories for TempFact train, dev and test articles
tempf_train_path = os.path.join(tempf_target_dir, 'train')
tempf_dev_path = os.path.join(tempf_target_dir, 'dev')
tempf_test_path = os.path.join(tempf_target_dir, 'test')

tempf_rkogu_path = os.path.join(tempf_target_dir, 'rkogu')
tempf_horisont_path = os.path.join(tempf_target_dir, 'horisont')

#os.mkdir(tempf_train_path)
#os.mkdir(tempf_dev_path)
#os.mkdir(tempf_test_path)

for text in tempf_train:
    filename = os.path.join(tempf_train_path, text.meta['filename']) + '.json'
    text_to_json(text, file=filename)
    
for text in tempf_dev:
    filename = os.path.join(tempf_dev_path, text.meta['filename']) + '.json'
    text_to_json(text, file=filename)
    
for text in tempf_test:
    filename = os.path.join(tempf_test_path, text.meta['filename']) + '.json'
    text_to_json(text, file=filename)
    
for text in tempf_rkogu:
    filename = os.path.join(tempf_rkogu_path, text.meta['filename']) + '.json'
    text_to_json(text, file=filename)
    
for text in tempf_horisont:
    filename = os.path.join(tempf_horisont_path, text.meta['filename']) + '.json'
    text_to_json(text, file=filename)